# ~ TilStat scraper ~

Utility to scrape the scores to enter the various courses over the years. Like any scrape utility, this code sucks.

In [1]:
import requests
from bs4 import BeautifulSoup as bs
from multiprocessing import Pool
import pandas as pd
import numpy as np

In [2]:
from datetime import datetime

## 2017/18 - 2020/21

In [3]:
base = 'http://didattica.polito.it'

In [4]:
indexes = []

for year in [2018,2019,2020,2021]:
    indexes += [
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc?p_a_acc={year}',
         'conting': False, 'course': 'Architettura', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc_conting?p_a_acc={year}',
         'conting': True, 'course': 'Architettura', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.ing?p_a_acc={year}',
         'conting': False, 'course': 'Ingegneria', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.ing_cont?p_a_acc={year}',
         'conting': True, 'course': 'Ingegneria', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des?p_a_acc={year}',
         'conting': False, 'course': 'Design', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des_cont?p_a_acc={year}',
         'conting': True, 'course': 'Design', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian?p_a_acc={year}',
         'conting': False, 'course': 'Pianificazione', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian_cont?p_a_acc={year}',
         'conting': True, 'course': 'Pianificazione', 'year':year}, 
    ]

In [5]:
# indexes = []

for year in [2022]:
    indexes += [
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc?p_a_acc={year}',
         'conting': False, 'course': 'Architettura', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.arc_conting?p_a_acc={year}',
         'conting': True, 'course': 'Architettura', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des?p_a_acc={year}',
         'conting': False, 'course': 'Design', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.des_cont?p_a_acc={year}',
         'conting': True, 'course': 'Design', 'year':year},

        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian?p_a_acc={year}',
         'conting': False, 'course': 'Pianificazione', 'year':year},
        {'uri':f'{base}/pls/portal30/preimma.pkg_grad.pian_cont?p_a_acc={year}',
         'conting': True, 'course': 'Pianificazione', 'year':year}, 
    ]

In [6]:
ranks = []
for index in indexes:
    html = requests.get(index['uri']).text
    soup = bs(html, 'html.parser')
    ranklinks = [link for link in soup.find_all("a")
                     if 'preimma' in link['href'] and link.text.startswith('Graduatoria del ')]
    for i, link in enumerate(ranklinks):
        
        # paranoid check
        # Es: Ingegneria has .ing string in each rank
        assert f".{index['course'].lower()[:3]}" in link['href'], "Wrong link assoc..."
        
        date = [int(x) for x in link.text.replace("Graduatoria del ","").split('/')]
        date = datetime(date[2],date[1],date[0])
        
        # paranoid check
        first_sane_rank = datetime(int(index['year'])-1, 7, 1) # dal primo luglio del primo anno di inizio coorte
        last_sane_rank = datetime(int(index['year']), 3, 15) # a metà marzo del secondo anno di inizio coorte
        assert first_sane_rank < date < last_sane_rank, "Wrong year assoc..."
        
        ranks.append({
            'uri': base+link['href'],
            'conting': index['conting'],
            'course': index['course'],
            'grad_n': i,
            'date': date
        })

In [7]:
ranks[0]

{'uri': 'http://didattica.polito.it/pls/portal30/preimma.pkg_grad.ing?p_data=05/09/2017&p_a_acc=2018',
 'conting': False,
 'course': 'Ingegneria',
 'grad_n': 0,
 'date': datetime.datetime(2017, 9, 5, 0, 0)}

In [18]:
x=0
def scrapeRank(rank):
    global x
    x+=1
    print(x)
    
    html = requests.get(rank['uri']).text
    soup = bs(html, 'html.parser')
    
    headers = soup.select("table th")
    rows = soup.select("table tr")
    cells = soup.select("table td")

    # paranoid check
    assert len(headers)*(len(rows)-1) == len(cells), "Wrong table format"

    headervalues = [str(h.text.lower().strip()) for h in headers]
    values = [str(c.text.strip()) for c in cells]
    shapedvalues = np.asarray(values).reshape((len(rows)-1),len(headers)).astype(str)
    df = pd.DataFrame(shapedvalues, columns=headervalues)

    df["totale"] = df["totale"].str.replace(',','.').astype(float)
    df['uri'] = rank['uri']
    df['conting'] = rank['conting']
    df['course'] = rank['course']
    df['grad_n'] = rank['grad_n']
    df['date'] = rank['date']
    

    
    return df

In [19]:
len(ranks)

182

In [20]:
dfs = list(map(scrapeRank, ranks))

1
2


KeyboardInterrupt: 

In [ ]:
result = pd.concat(dfs, ignore_index=True)

In [ ]:
result.to_csv(f"backup17181921_{str(datetime.now()).replace(':','-')}.csv")